# 3. Эксперименты и проверка результатов

Июль–октябрь 2022 уже известны по предыдущей оценке: статус оценки исследовательский. Все модели фиксируются до этой оценки. Финальная фаза запрещает fit.

In [1]:
from pathlib import Path
import json, sys
import numpy as np
import pandas as pd
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "src/research/feature_automation").is_dir())
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
RUN_TRAINING = False
RESULT_DIR = ROOT / ("results/runs/notebook-reproduction" if RUN_TRAINING else "results/final")
DATA_DIR = ROOT / "data/processed"
read = lambda name: json.loads((RESULT_DIR / name).read_text())
print("Результаты фиксированного эксперимента; RUN_TRAINING =", RUN_TRAINING)


Результаты фиксированного эксперимента; RUN_TRAINING = False


In [2]:
if RUN_TRAINING:
    import subprocess
    subprocess.run([sys.executable, "-m", "research.feature_automation.experiment",
                    "--data-dir", str(DATA_DIR), "--out", str(RESULT_DIR), "--phase", "evaluate"],
                   cwd=ROOT, check=True)
else:
    print("Этап evaluate ранее выполнен CLI; ниже его журнал и результаты.")


Этап evaluate ранее выполнен CLI; ниже его журнал и результаты.


In [3]:
e = read("evaluation.json")
print(pd.DataFrame(e["metrics"]).T.to_string())
print("Статус:", e["status"], "число fit при оценке:", e["evaluation_fits"])

                       Recall@5    NDCG@5  Precision@5  Precision@5_all      rows  positives  eligible_cases  all_cases   LogLoss   ROC-AUC
old_5                  0.591759  0.510016     0.225424           0.0266  100000.0      508.0           236.0     2000.0  0.019692  0.937077
manual                 0.655309  0.561431     0.252542           0.0298  100000.0      508.0           236.0     2000.0  0.018243  0.953205
automatic_all          0.636841  0.558906     0.247458           0.0292  100000.0      508.0           236.0     2000.0  0.018388  0.953618
automatic_selected     0.643181  0.563538     0.247458           0.0292  100000.0      508.0           236.0     2000.0  0.019283  0.925914
manual_plus_automatic  0.643938  0.563557     0.248305           0.0293  100000.0      508.0           236.0     2000.0  0.018321  0.953330
pair_popularity        0.617059  0.530301     0.235593           0.0278  100000.0      508.0           236.0     2000.0       NaN       NaN
popularity          

In [4]:
print(json.dumps(e["bootstrap"], ensure_ascii=False, indent=2))
print("Месяцы:")
for name in ("manual", "automatic_selected", "manual_plus_automatic", "pair_popularity"):
    print(name, pd.DataFrame(e["monthly"][name]).to_string(index=False))

{
  "manual_plus_automatic - manual": {
    "delta": -0.011370056497175143,
    "ci95": [
      -0.02924013620549264,
      0.004185701391101199
    ],
    "repeats": 5000,
    "empty_draws": 0,
    "seed": 20261013,
    "scope": "fixed fitted models and observed months; client sampling only"
  },
  "automatic_selected - manual": {
    "delta": -0.01212750636977955,
    "ci95": [
      -0.038387646237328724,
      0.013066908148132496
    ],
    "repeats": 5000,
    "empty_draws": 0,
    "seed": 20261013,
    "scope": "fixed fitted models and observed months; client sampling only"
  },
  "manual - pair_popularity": {
    "delta": 0.0382491414644954,
    "ci95": [
      0.001742630747123814,
      0.07507538737298086
    ],
    "repeats": 5000,
    "empty_draws": 0,
    "seed": 20261013,
    "scope": "fixed fitted models and observed months; client sampling only"
  },
  "manual_plus_automatic - pair_popularity": {
    "delta": 0.02687908496732026,
    "ci95": [
      -0.0124501892017588

In [5]:
print("Срезы:", json.dumps(e["groups"], ensure_ascii=False, indent=2))
print("Покрытие:", e["coverage"])
print(pd.DataFrame(read("benchmark.json")).to_string(index=False))

Срезы: {
  "repeat_3m": {
    "old_5": {
      "Recall@5": 0.8850615802108339,
      "NDCG@5": 0.7971064245747366,
      "Precision@5": 0.3104477611940298,
      "Precision@5_all": 0.0208,
      "rows": 100000,
      "positives": 280,
      "eligible_cases": 134,
      "all_cases": 2000,
      "LogLoss": null,
      "ROC-AUC": null
    },
    "manual": {
      "Recall@5": 0.9016245984529566,
      "NDCG@5": 0.8010411217742058,
      "Precision@5": 0.31940298507462683,
      "Precision@5_all": 0.021400000000000002,
      "rows": 100000,
      "positives": 280,
      "eligible_cases": 134,
      "all_cases": 2000,
      "LogLoss": null,
      "ROC-AUC": null
    },
    "automatic_all": {
      "Recall@5": 0.8751735205093413,
      "NDCG@5": 0.7850509914434741,
      "Precision@5": 0.3149253731343284,
      "Precision@5_all": 0.0211,
      "rows": 100000,
      "positives": 280,
      "eligible_cases": 134,
      "all_cases": 2000,
      "LogLoss": null,
      "ROC-AUC": null
    },
    "

## Независимый пересчёт

Ниже ranking-метрики пересчитываются отдельным циклом по множествам купленных активов. Прежние оценки и веса не меняются.

In [6]:
from research.feature_automation.verify import metric_oracle
saved = pd.read_csv(RESULT_DIR / "predictions.csv.gz", dtype={"customerID":str,"ISIN":str},
                    parse_dates=["prediction_date"], float_precision="round_trip")
for name, metrics in e["metrics"].items():
    secondary = saved.pair_popularity_secondary.to_numpy() if name == "pair_popularity" else None
    oracle = metric_oracle(saved, saved[name].to_numpy(), secondary)
    assert max(abs(value-metrics[key]) for key,value in oracle.items()) < 1e-12
print("Независимый пересчёт семи методов совпал.")
print(json.dumps(read("verification.json"), indent=2))

Независимый пересчёт семи методов совпал.
{
  "unit_tests": 8,
  "metric_oracle_errors": {
    "old_5": {
      "Recall@5": 0.0,
      "NDCG@5": 0.0,
      "Precision@5": 0.0,
      "Precision@5_all": 0.0,
      "LogLoss": 6.938893903907228e-18,
      "ROC-AUC": 0.0
    },
    "manual": {
      "Recall@5": 1.1102230246251565e-16,
      "NDCG@5": 0.0,
      "Precision@5": 0.0,
      "Precision@5_all": 0.0,
      "LogLoss": 3.469446951953614e-18,
      "ROC-AUC": 1.1102230246251565e-16
    },
    "automatic_all": {
      "Recall@5": 0.0,
      "NDCG@5": 1.1102230246251565e-16,
      "Precision@5": 0.0,
      "Precision@5_all": 0.0,
      "LogLoss": 6.938893903907228e-18,
      "ROC-AUC": 0.0
    },
    "automatic_selected": {
      "Recall@5": 0.0,
      "NDCG@5": 1.1102230246251565e-16,
      "Precision@5": 0.0,
      "Precision@5_all": 0.0,
      "LogLoss": 3.469446951953614e-18,
      "ROC-AUC": 1.1102230246251565e-16
    },
    "manual_plus_automatic": {
      "Recall@5": 0.0,
      

## Вывод

Эффект автоматизации определяется фактической разностью и интервалом. Покупки не доказывают прибыльность рекомендаций. Bootstrap условен на периоде и выбранных моделях; независимое подтверждение требует нового закрытого периода.